# Table 1 Creation

This notebook generates the Table1 based on the matched case/controls and their corresponding NLP extracted home locations and body positions. This notebook does the grunt of the actual analytical work for the project.

In [104]:
import pandas as pd
import orjson
import itertools
from tableone import TableOne

In [105]:
case_controls = pd.read_csv("../data/processed/matched_case_controls.csv")
case_controls.drop(
    columns=["matchnumber", "distance", "weights"],
    inplace=True,
)
print(case_controls.shape)
case_controls.head()

(606, 6)


,ccmeo_case,age_category,race_category,gender_category,year_category,case_control_category
0,IN2019-00415,25-34,White,Male,2019,1
1,IN2019-00615,35-44,White,Female,2019,0
2,IN2020-01872,55-64,White,Male,2020,0
3,in2019-00423,35-44,White,Female,2019,1
4,IN2019-01542,35-44,Black,Male,2019,0


In [106]:
case_mask = case_controls["case_control_category"] == 1
case_ids = case_controls.loc[case_mask, "ccmeo_case"].unique().tolist()
control_ids = case_controls.loc[~case_mask, "ccmeo_case"].unique().tolist()
print(f"Number of (carfentanil) cases: {len(case_ids)}")
print(f"Number of (fentanyl) controls: {len(control_ids)}")

Number of (carfentanil) cases: 202
Number of (fentanyl) controls: 404


In [107]:
with open("../data/processed/metamap_results_parsed.jsonl", "rb") as f:
    nlp_data = [orjson.loads(l) for l in f]

nlp_df = pd.DataFrame(nlp_data)
names = nlp_df["name"].dropna().unique().tolist()
print(f"Number of unique names: {len(names)}")
nlp_df[names] = pd.get_dummies(nlp_df["name"])
categories = nlp_df["category"].dropna().unique().tolist()
print(f"Number of unique categories: {len(categories)}")
nlp_df[categories] = pd.get_dummies(nlp_df["category"])
nlp_df.drop(columns=["name", "sui", "category"], inplace=True)
nlp_df.rename(columns={"identifier": "ccmeo_case"}, inplace=True)
nlp_grouped_by_case = nlp_df.groupby("ccmeo_case").sum().reset_index()
print(f"ccmeo_case is unique: {nlp_grouped_by_case['ccmeo_case'].is_unique}")
nlp_grouped_by_case.head()

Number of unique names: 53
Number of unique categories: 2
ccmeo_case is unique: True


,ccmeo_case,Bedroom,Bathroom,Sitting position,Dining room,Supine Position,Kitchen,Prone Position,Upper floor,Lying in bed,...,Arm position finding,Lounge environment,Seventh floor,Sixth floor,Room of building,Fourth floor,Workshop building,Prone-kneeling,home premises,body position
0,CA2019-00103,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
1,CA2019-00377,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,CA2021-00165,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,CA2021-00278,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,CA2021-00438,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [108]:
merged = case_controls.merge(nlp_grouped_by_case, on="ccmeo_case", how="left")
print(merged.shape)

(606, 61)


In [111]:
combined = merged.copy()
positions = {
    "sitting": [
        "Sitting position",
        "Side-sitting",
        "Sitting upright",
    ],
    "standing": [
        "Standing in water",
        "Standing position",
    ],
    "kneeling": [
        "Kneeling position",
        "Kneeling (finding)",
    ],
    "lying": [
        "Supine Position",
        "Prone Position",
        "Lying in bed",
        "Left lateral decubitus position",
        "Right lateral decubitus position",
        "Lateral decubitus position",
        "Recumbent body position",
        "Prone-kneeling",
        "Frog-like posture",
    ],
    "unknown_position": [
        "Arm position finding",
        "Head position finding",
    ],
}
locations = {
    "inside": [
        "Bedroom",
        "Bathroom",
        "Dining room",
        "Kitchen",
        "Shower room",
        "Sitting room",
        "Attic Room",
        "Bed area",
        "Side room",
        "Lounge environment",
        "Room of building",
    ],
    "outside": [
        "Landing",
        "Back yard",
        "Front yard",
        "Garage",
        "Garden lawn",
        "Workshop building",
    ],
    "unknown_location": [
        "Fourth floor",
        "Sixth floor",
        "Seventh floor",
        "Ninth floor",
        "Ground floor",
        "Eighth floor",
        "Top of staircase",
        "Tenth floor",
        "Downstairs",
        "First floor",
        "Third floor",
        "Bottom of staircase",
        "Second floor",
        "Upstairs",
        "Hallway",
        "Upper floor",
        "Basement floor",
        "Fifth floor",
    ],
}
columns_defined = set()
for k, v in positions.items():
    columns_defined.update(v)
for k, v in locations.items():
    columns_defined.update(v)
diff = set(combined.columns) - columns_defined
assert diff == set(["home premises", "body position", "age_category", "case_control_category", "ccmeo_case", "gender_category", "race_category", "year_category"]), diff

for k, v in itertools.chain(positions.items(), locations.items()):
    combined[k] = combined[v].any(axis=1)

combined.drop(columns=list(columns_defined) + ["home premises", "body position"], inplace=True)
print(combined.shape)
combined.head()

(606, 14)


,ccmeo_case,age_category,race_category,gender_category,year_category,case_control_category,sitting,standing,kneeling,lying,unknown_position,inside,outside,unknown_location
0,IN2019-00415,25-34,White,Male,2019,1,False,False,False,True,False,True,False,True
1,IN2019-00615,35-44,White,Female,2019,0,False,False,False,True,False,False,False,False
2,IN2020-01872,55-64,White,Male,2020,0,False,False,False,False,False,True,False,True
3,in2019-00423,35-44,White,Female,2019,1,False,False,False,False,False,False,False,False
4,IN2019-01542,35-44,Black,Male,2019,0,False,False,False,False,False,False,False,False


In [114]:
cols = list(combined.columns)
cols.remove("ccmeo_case")
cols.remove("case_control_category")
tbl = TableOne(
    combined,
    groupby="case_control_category",
    columns=cols,
    categorical=cols,
    overall=True,
    pval=True,
)
tbl.to_excel("../data/output/matched_table1.xlsx")
print("Table 1: Carfenatil vs. Fentanyl")
print(tbl.tabulate(tablefmt="github"))

Table 1: Carfenatil vs. Fentanyl
|                         |        | Missing   | Overall    | 0          | 1          | P-Value   |
|-------------------------|--------|-----------|------------|------------|------------|-----------|
| n                       |        |           | 606        | 404        | 202        |           |
| age_category, n (%)     | 18-24  | 0         | 25 (4.1)   | 20 (5.0)   | 5 (2.5)    | 0.492     |
|                         | 25-34  |           | 129 (21.3) | 90 (22.3)  | 39 (19.3)  |           |
|                         | 35-44  |           | 143 (23.6) | 93 (23.0)  | 50 (24.8)  |           |
|                         | 45-54  |           | 130 (21.5) | 81 (20.0)  | 49 (24.3)  |           |
|                         | 55-64  |           | 148 (24.4) | 101 (25.0) | 47 (23.3)  |           |
|                         | 65+    |           | 31 (5.1)   | 19 (4.7)   | 12 (5.9)   |           |
| race_category, n (%)    | Asian  | 0         | 3 (0.5)    | 2 (0.